In [ ]:
from elos.elo_tracker import EloTracker
from utils.generic_utils import load_all_games_csv, get_teams, basic_win_prob
from simulation.simulation import simulate_seasons
import pandas as pd

# Simulation

This notebook focuses on simulating the 2025 season using Elo ratings, estimating each team's probability of making it to the playoffs and different rounds within them.

## Get Divisions and Leagues for Past Season Teams

In [3]:
# Get divisions and leagues for each team
AL_EAST = {'TOR', 'NYA', 'BOS', 'TBA', 'BAL'}
AL_CENTRAL = {'CLE', 'DET', 'KCA', 'MIN', 'CHA'}
AL_WEST = {'SEA', 'HOU', 'TEX', 'ATH', 'ANA'}

AL = [AL_EAST, AL_CENTRAL, AL_WEST]

NL_EAST = {'PHI', 'NYN', 'MIA', 'ATL', 'WAS'}
NL_CENTRAL = {'MIL', 'CHN', 'CIN', 'SLN', 'PIT'}
NL_WEST = {'LAN', 'SDN', 'SFN', 'ARI', 'COL'}

NL = NL_EAST, NL_CENTRAL, NL_WEST

CURRENT_TEAMS = AL_EAST | AL_CENTRAL | AL_WEST | NL_EAST | NL_CENTRAL | NL_WEST

In [4]:
PAST_SEASON_TEAMS = CURRENT_TEAMS.copy()
# A's moved
PAST_SEASON_TEAMS.discard('ATH')
PAST_SEASON_TEAMS.add('OAK')

## Get Elo Ratings for each team

In [5]:
all_games = load_all_games_csv('../data/gameinfo_cleaned.csv')

teams = get_teams(all_games)
et = EloTracker(teams)

et.add_history(all_games)

# Get dict of latest elos for past teams
elos_map = {team: et.elos_map[team][-1][3] for team in PAST_SEASON_TEAMS}

# Revert to mean by 1/3 for new season
elos_map = {team: elos_map[team] + (1500 - elos_map[team]) / 3 for team in PAST_SEASON_TEAMS}

# Change OAK to ATH for A's move
elos_map['ATH'] = elos_map['OAK']
del elos_map['OAK']

elos_map

/Users/lancehendricks/Documents/College Coding/ML/Elo Ratings/analysis/src/utils/utils.py:27: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(filename)


{'CHA': 1432.28608563123,
 'CLE': 1510.3684505749704,
 'KCA': 1491.9308675960867,
 'CIN': 1491.346298436719,
 'ARI': 1514.5682422324635,
 'SFN': 1498.8203286993905,
 'TBA': 1509.6581249129413,
 'NYN': 1516.1993638221697,
 'MIA': 1474.8425358668817,
 'BOS': 1498.0906207045605,
 'DET': 1505.9052275046763,
 'SDN': 1523.9338488249748,
 'MIL': 1520.8962222440791,
 'BAL': 1517.1047622196943,
 'ATL': 1522.6776363652439,
 'NYA': 1521.865698858941,
 'HOU': 1520.0027065050556,
 'SEA': 1510.3629520246325,
 'SLN': 1501.475780378102,
 'PIT': 1486.1727607963771,
 'WAS': 1476.3742822615345,
 'LAN': 1543.4531570264198,
 'COL': 1462.099952940796,
 'MIN': 1498.029487442219,
 'ANA': 1464.2556058241114,
 'TOR': 1495.5159210103802,
 'TEX': 1498.5358956818375,
 'CHN': 1502.8983221771189,
 'PHI': 1522.731826710146,
 'ATH': 1467.6783484973278}

## Get Schedule

In [6]:
schedule_df = pd.read_csv('../data/2025schedule.csv')
schedule_df.head()

,Date,Num,Day,Visitor,League,Game,Home,League.1,Game.1,Day/Night,Location,Postponed,Makeup
0,20250318,0,Tuesday,LAN,NL,1,CHN,NL,1,n,TOK01,NaN,NaN
1,20250319,0,Wednesday,LAN,NL,2,CHN,NL,2,n,TOK01,NaN,NaN
2,20250327,0,Thursday,MIL,NL,1,NYA,AL,1,d,NYC21,NaN,NaN
3,20250327,0,Thursday,BAL,AL,1,TOR,AL,1,d,TOR02,NaN,NaN
4,20250327,0,Thursday,BOS,AL,1,TEX,AL,1,d,ARL03,NaN,NaN


In [7]:
# Get as list
schedule = [(home, away) for home, away in zip(schedule_df['Home'], schedule_df['Visitor'])]

## Simulate Seasons

In [8]:
team_results = simulate_seasons(10000, schedule, AL, NL, elos_map, elo_prob_func=lambda home_elo, away_elo, game_info: basic_win_prob(home_elo + 28, away_elo))

100%|██████████| 10000/10000 [01:19<00:00, 126.53it/s]


In [9]:
# Get DF of results
avg_wins = [team_results[team][0] for team in CURRENT_TEAMS]
playoff_pcts = [team_results[team][1] for team in CURRENT_TEAMS]
div_pcts = [team_results[team][2] for team in CURRENT_TEAMS]
champ_pcts = [team_results[team][3] for team in CURRENT_TEAMS]
ws_pcts = [team_results[team][4] for team in CURRENT_TEAMS]
winner_pcts = [team_results[team][5] for team in CURRENT_TEAMS]

# Make PD dataframe to easily visualize
results_df = pd.DataFrame({'Team':list(CURRENT_TEAMS), 'Avg. Wins':avg_wins, 'Playoff %':playoff_pcts,
                           'Divisional %':div_pcts, 'Championship %':champ_pcts, 'WS %':ws_pcts, 'Win WS %':winner_pcts})
results_df = results_df.sort_values(by='Win WS %', ascending=False).reset_index(drop=True)
results_df

,Team,Avg. Wins,Playoff %,Divisional %,Championship %,WS %,Win WS %
0,LAN,91.2276,79.01,56.29,30.91,16.19,8.51
1,NYA,86.4095,60.67,44.62,24.18,13.13,7.52
2,HOU,86.6215,65.99,43.52,22.42,11.35,6.32
3,BAL,85.1136,57.60,40.38,20.90,10.36,6.10
4,CLE,84.3802,56.11,42.81,22.88,11.18,5.95
5,MIL,85.9784,59.50,45.74,24.55,12.59,5.79
6,ATL,86.2389,61.03,43.17,22.72,11.53,5.19
7,DET,83.4058,51.63,37.71,20.06,9.80,5.15
8,PHI,86.3485,59.69,41.60,21.73,11.27,4.91
9,TBA,83.1702,45.71,30.06,14.70,7.49,4.41
